# Decision Tree Classifier

---

## Overview

A **decision tree** recursively partitions the feature space using binary splits. At each node, we choose the feature $j$ and threshold $t$ that maximizes the **information gain**:

$$\text{Gain}(j, t) = \text{Gini}(\text{parent}) - \frac{N_L}{N}\,\text{Gini}(\text{left}) - \frac{N_R}{N}\,\text{Gini}(\text{right})$$

The **Gini impurity** of a node with class distribution $\{p_k\}$ is:

$$\text{Gini} = 1 - \sum_{k} p_k^2$$

A **leaf node** returns the majority class of its training samples.

---

**Dataset:** Obesity Levels (`Obesity_levels.csv`). Falls back to sklearn iris if `FileNotFoundError`.

**Task:** Classify obesity category from health/lifestyle features.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sns.set_theme()

from rice_ml.supervised_learning import DecisionTreeClassifier
from rice_ml.preprocess import train_test_split
from rice_ml.metrics import accuracy_score, confusion_matrix


In [ ]:
try:
    import pandas as pd
    df = pd.read_csv('../../data/Obesity_levels.csv')
    from rice_ml.preprocess import OrdinalEncoder
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    for col in cat_cols:
        enc = OrdinalEncoder()
        df[col] = enc.fit_transform(df[[col]]).ravel()
    target_col = 'NObeyesdad' if 'NObeyesdad' in df.columns else df.columns[-1]
    X = df.drop(columns=[target_col]).values.astype(float)
    y = df[target_col].values.astype(int)
    class_names = [str(c) for c in np.unique(y)]
    print(f'Loaded obesity dataset: {X.shape}')
except FileNotFoundError:
    from sklearn.datasets import load_iris
    iris = load_iris()
    X, y = iris.data, iris.target
    class_names = list(iris.target_names)
    print('CSV not found. using iris dataset (3 classes)')


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Effect of `max_depth` on Accuracy

Deeper trees fit training data better but may overfit. We compare train vs test accuracy across depths.

In [ ]:
depths = range(1, 11)
train_accs, test_accs = [], []

for d in depths:
    clf = DecisionTreeClassifier(max_depth=d)
    clf.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, clf.predict(X_train)))
    test_accs.append(accuracy_score(y_test, clf.predict(X_test)))

plt.figure(figsize=(10, 6))
plt.plot(depths, train_accs, marker='o', label='Train', color='steelblue')
plt.plot(depths, test_accs, marker='s', label='Test', color='salmon')
plt.xlabel('max_depth', fontsize=15)
plt.ylabel('Accuracy', fontsize=15)
plt.title('Decision Tree: Depth vs Accuracy', fontsize=18)
plt.legend(fontsize=13)
plt.show()

In [ ]:
best_depth = depths[np.argmax(test_accs)]
clf = DecisionTreeClassifier(max_depth=best_depth)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(f'Best depth: {best_depth}')
print(f'Test Accuracy: {accuracy_score(y_test, y_pred):.4f}')

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted', fontsize=13)
plt.ylabel('True', fontsize=13)
plt.title(f'Decision Tree (depth={best_depth}): Confusion Matrix', fontsize=16)
plt.show()

## Interpretation

- A **depth-1 tree** (decision stump) makes a single split. low complexity, often low accuracy.
- As depth increases, training accuracy rises toward 1.0 but test accuracy may plateau or decrease (overfitting).
- The optimal depth balances bias and variance. use cross-validation in practice to find it.